# 11.10 切得更細，圖片會佔多少序列空間？


16×16 圖把塊邊長從 4 改成 2，上下、左右塊數都加倍，總位置從 16 成 64。更細的材料可能保住小物件，但也佔更多上下文。先算圖片與二十個文字位置合起來多長。

這裡也數完整注意力表的格子：每列是一個要讀取資訊的位置，每欄是一個提供資訊的位置，兩邊都包含圖片與文字位置。總長若是 N，每個讀取位置對每個來源位置各有一格，因此有 N×N 格；程式的 `total**2` 就是在數這張完整表。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
height = 16
text_tokens = 20
for patch_size in [8, 4, 2]:
    visual_tokens = (height // patch_size) ** 2
    total = text_tokens + visual_tokens
    print("邊長", patch_size, "視覺位置", visual_tokens, "總長度", total, "注意力格數", total**2)

邊長 8、4、2 分別有 4、16、64 個視覺位置；總長 24、36、84，完整注意力矩陣格數 576、1296、7056。視覺 token 在這裡是一塊圖的特徵位置，不是一個中文字；平方格數是成本線索，不等於精確秒數。

若模型上限 64，最後方案的 84 已放不下。截掉文字可能連答案目標一起截掉；設定更大上限也不保證相關任務已學好。需要同時看容量、可見材料與答案。

改切法還會改入口矩陣：RGB 的邊長 4 塊有 48 個值，邊長 2 只有 12。舊 48 欄權重不能直接接新 12 值，必須用對應配置與訓練，不能只改推論參數便比較效果。

既有小對照只做塊邊長 4 與 8，兩者最後圖片題都是 9/12，沒有測出細切優勢，也沒測小字。較大塊位置少，但每塊像素多，入口矩陣反而較大；位置成本與參數量是不同帳目。

練習圖邊長改 32，三種位置成 16、64、256；最後總長 276、注意力格數 76176。它幫助先規劃可用空間，並未證明增加位置就增加理解。

<details>
<summary>回顧與查證</summary>

可回顧：[10.2切塊數量](https://birdhackor.github.io/tiny-perceptron-vlm/10.2.html)、[3.1多位置注意力](https://birdhackor.github.io/tiny-perceptron-vlm/3.1.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/vision_ablation.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
